In [ ]:
#| default_exp wfbuild

In [ ]:
#| hide
from fastcore.test import *
from nbdev.showdoc import *
from gheasy.workflow import Workflow
from tempfile import TemporaryDirectory
import yaml
def parse_wf(y):
    "A composed workflow as data. pyyaml reads the unquoted key `on` as the boolean `True`."
    d = yaml.safe_load(y)
    if True in d: d['on'] = d.pop(True)
    return d
def built(wfb): return parse_wf(wfb.build().to_yaml())
def on_of(on):
    "The `on:` block the trigger half of a spec renders to."
    wfb = Workflow('ci'); _triggers(wfb, on)
    return built(wfb)['on']

Building a workflow by choosing, not by typing YAML.

`TRIGGERS` and `PRESETS` are the vocabulary a picker offers. A spec is what was picked, as JSON.
`compose` turns one into the YAML that goes in `.github/workflows`.

Every trigger, job and step is built through gheasy's `Workflow` DSL. Nothing here writes YAML by
hand and nothing here touches the network. `workflows` reads what this module writes.

In [ ]:
#| export
from __future__ import annotations

In [ ]:
#| export
import json

In [ ]:
#| export
from fastcore.all import Path

In [ ]:
#| export
SPEC_DIR = 'workflows'
_spec_dir = SPEC_DIR

def use_spec_dir(dir):
    "Name the directory a spec is kept in, relative to the repository root."
    global _spec_dir
    _spec_dir = str(dir)

def spec_dir(): return _spec_dir

A spec is kept at `<root>/workflows/<name>.json`, beside the repository root. The YAML it built
goes to `.github/workflows`, which is `WORKFLOW_DIR` in `workflows`.

A host that keeps its specs somewhere else says so once with `use_spec_dir`, and `spec_path`,
`load_spec` and `save_spec` read it as they are called. Each also takes the directory as a trailing
argument, which is what a caller wrapping them for one repository passes.

In [ ]:
#| export
TRIGGERS = [
    {'id': 'push', 'label': 'on push', 'kind': 'branches', 'default': 'main',
        'doc': 'Every push to these branches.'},
    {'id': 'tags', 'label': 'on a tag', 'kind': 'branches', 'default': 'v*',
        'doc': 'Every push of a matching tag. `v*` is what `ship-release` pushes.'},
    {'id': 'pull_request', 'label': 'on pull request', 'kind': 'branches', 'default': 'main',
        'doc': 'Every pull request targeting these branches.'},
    {'id': 'workflow_dispatch', 'label': 'run button', 'kind': 'flag', 'default': True,
        'doc': 'Adds a Run button, here and on GitHub. Worth having on almost everything.'},
    {'id': 'release', 'label': 'on release', 'kind': 'flag', 'default': False,
        'doc': 'When a GitHub release is published — what a PyPI publish job waits for.'},
    {'id': 'schedule', 'label': 'on a schedule', 'kind': 'cron', 'default': '0 9 * * 1',
        'doc': 'A cron expression in UTC. `0 9 * * 1` is Mondays at 09:00.'},
]

In [ ]:
#| export
PRESETS = [
    {'id': 'uv_test', 'job': 'test', 'name': 'Test (uv)',
        'doc': 'Checkout, install with uv, run the tests.',
        'options': [
            {'key': 'test_cmd', 'label': 'test command', 'type': 'text', 'default': 'pytest',
                'doc': 'Run through `uv run`, so `pytest -x` becomes `uv run pytest -x`.'},
            {'key': 'python_versions', 'label': 'python versions', 'type': 'list', 'default': '',
                'doc': 'Comma separated. More than one adds a matrix; empty runs once.'}]},
    {'id': 'uv_lint', 'job': 'lint', 'name': 'Lint (ruff)',
        'doc': 'ruff check and a formatting check.',
        'options': [{'key': 'lint_cmd', 'label': 'lint command', 'type': 'text',
            'default': 'uv run ruff check . && uv run ruff format --check .', 'doc': ''}]},
    {'id': 'nbdev_test', 'job': 'test', 'name': 'Test (nbdev)',
        'doc': 'nbdev_test, and a check that the notebooks are clean — what breaks in an nbdev repo.',
        'options': []},
    {'id': 'uv_pypi', 'job': 'publish', 'name': 'Publish to PyPI',
        'doc': 'Build and publish through trusted publishing. Only runs on a release event, so '
        'pair it with the release trigger.',
        'options': []},
    {'id': 'fastship', 'job': 'release', 'name': 'Release (fastship)',
        'doc': 'What `ship-release` pushes a `v*` tag for: build it, write the notes, publish it '
        'through trusted publishing. Pair it with a push trigger on `v*` tags.',
        'options': []},
    {'id': 'docker', 'job': 'docker', 'name': 'Docker image',
        'doc': 'Build and push to GHCR as :latest.',
        'options': [{'key': 'app_name', 'label': 'image name', 'type': 'text', 'default': '',
            'doc': 'Defaults to the project name.'}]},
    {'id': 'node', 'job': 'frontend', 'name': 'Node build', 'doc': 'npm ci, build, test.',
        'options': []},
    {'id': 'rust', 'job': 'build', 'name': 'Rust build', 'doc': 'cargo build, test, clippy.',
        'options': []},
    {'id': 'go', 'job': 'build', 'name': 'Go build', 'doc': 'go build, test, vet.', 'options': []},
    {'id': 'deploy', 'job': 'deploy', 'name': 'Deploy to a VPS',
        'doc': 'Every key in env_schema as a secret or a variable, the deploy key, then your '
        'deploy command. The shape lego uses.',
        'options': [
            {'key': 'cmd', 'label': 'deploy command', 'type': 'text', 'default': 'python deploy.py deploy',
                'doc': ''},
            {'key': 'lfs', 'label': 'checkout LFS files', 'type': 'flag', 'default': True, 'doc': ''}]},
    {'id': 'run', 'job': 'run', 'name': 'Anything else',
        'doc': 'Checkout, install with uv, then the commands you give it — one per line.',
        'options': [
            {'key': 'job_id', 'label': 'job id', 'type': 'text', 'default': 'run',
                'doc': 'What other jobs name in `needs`.'},
            {'key': 'cmds', 'label': 'commands', 'type': 'lines', 'default': 'echo hello',
                'doc': 'One step per line.'},
            {'key': 'install', 'label': 'install dependencies first', 'type': 'flag', 'default': True,
                'doc': ''}]},
]

`TRIGGERS` and `PRESETS` are the two lists a picker renders and `compose` dispatches on.

A trigger's `kind` says what the picker asks for: `branches` a comma separated list, `cron` a cron
expression, `flag` a checkbox. A preset's `job` is the id its job takes when the spec names none.
Its `options` are the fields the picker shows, and an option's `type` is one of `text`, `list`,
`lines` and `flag`.

A preset added here and not added to `compose` is a row the picker offers and no job built.

In [ ]:
#| export
def preset_rows():
    "The vocabulary, as the panel renders it."
    return dict(presets=PRESETS, triggers=TRIGGERS)

In [ ]:
rows = preset_rows()
[p['id'] for p in rows['presets']], [t['id'] for t in rows['triggers']]

(['uv_test',
  'uv_lint',
  'nbdev_test',
  'uv_pypi',
  'fastship',
  'docker',
  'node',
  'rust',
  'go',
  'deploy',
  'run'],
 ['push', 'tags', 'pull_request', 'workflow_dispatch', 'release', 'schedule'])

In [ ]:
#| export
def _listy(value):
    if isinstance(value, (list, tuple)): return [str(v).strip() for v in value if str(v).strip()]
    return [v.strip() for v in str(value or '').split(',') if v.strip()]

In [ ]:
#| export
def _lines(value):
    if isinstance(value, (list, tuple)): return [str(v) for v in value if str(v).strip()]
    return [l for l in str(value or '').splitlines() if l.strip()]

In [ ]:
#| export
def _branches(value):
    "The branches a push or pull_request trigger names; `True` and empty both mean `main`."
    return _listy('main' if value is True else value) or ['main']

Three coercions between what a form returns and what gheasy takes.

`_listy` splits a comma separated string and takes a list as it is. Both forms drop empty entries
and strip each one. `_lines` splits on newlines and keeps each line as written, indentation
included, so a command is not reformatted on its way into a step.

`_branches` answers `['main']` for an empty field and for a ticked checkbox. A push or pull_request
trigger never renders with no branches at all.

In [ ]:
_listy('main, dev ,'), _lines('echo one\n\n  echo two'), _branches(True)

(['main', 'dev'], ['echo one', '  echo two'], ['main'])

In [ ]:
#| hide
test_eq(_listy(['a', '  ', ' b ']), ['a', 'b'])
test_eq(_listy(None), [])
test_eq(_lines(['keep me', ' ']), ['keep me'])
test_eq(_lines(None), [])
test_eq(_branches(''), ['main'])
test_eq(_branches('dev, main'), ['dev', 'main'])

In [ ]:
#| export
def _triggers(wfb, on):
    "Apply the trigger part of a spec, whose one shape gheasy renders into `on:`'s three."
    used = False
    push, tags = on.get('push'), on.get('tags')
    # gheasy renders one `push:` block, so branches and tags go into the same call.
    if push not in (None, False) or tags not in (None, False):
        wfb.on.push(branches=_branches(push) if push not in (None, False) else None,
            tags=_listy('v*' if tags is True else tags) or None)
        used = True
    pr = on.get('pull_request')
    if pr not in (None, False):
        wfb.on.pull_request(branches=_branches(pr))
        used = True
    if on.get('workflow_dispatch'):
        wfb.on.workflow_dispatch()
        used = True
    if on.get('release'):
        wfb.on.release(types=['published'])
        used = True
    cron = on.get('schedule')
    if cron:
        wfb.on.schedule(str(cron).strip())
        used = True
    if not used: wfb.on.workflow_dispatch()

A spec has one trigger shape. `on:` has three, and gheasy renders whichever fits what was asked
for.

`push` and `tags` are two rows in the picker and one `push:` block in the file. Asking for both
gives one block carrying `branches` and `tags`.

A missing key and `False` both mean off. A spec that turns every trigger off still gets
`workflow_dispatch`, so every workflow this builds can be started by hand.

In [ ]:
on_of({'push': 'main, dev', 'tags': 'v*'})

{'push': {'branches': ['main', 'dev'], 'tags': ['v*']}}

One trigger with no filters renders as a bare string rather than a mapping. That is the third
shape `Workflows._triggers` reads back.

In [ ]:
on_of({'push': False, 'workflow_dispatch': False})

'workflow_dispatch'

In [ ]:
#| hide
test_eq(on_of({'push': True}), {'push': {'branches': ['main']}})
test_eq(on_of({'tags': True}), {'push': {'tags': ['v*']}})
test_eq(on_of({'pull_request': 'main'}), {'pull_request': {'branches': ['main']}})
test_eq(on_of({'release': True}), {'release': {'types': ['published']}})
test_eq(on_of({'schedule': '0 9 * * 1'}), {'schedule': [{'cron': '0 9 * * 1'}]})
for t in TRIGGERS:                                  # every id in the picker reaches a trigger
    on = on_of({'push': 'main', t['id']: t['default'] or True})
    test_eq(len(on), 1 if t['id'] in ('push', 'tags') else 2)

In [ ]:
#| export
def deploy_job(wfb, job_id, opts, schema, app):
    "lego's deploy job: the schema as `env:`, the key, then the command."
    env = {k: (f'${{{{ secrets.{k} }}}}' if v is None else f'${{{{ vars.{k} }}}}')
        for k, v in (schema or {}).items()}
    env['DEPLOY_KEY'] = '${{ secrets.DEPLOY_KEY }}'
    ssh = ('mkdir -p ~/.ssh && name="${SERVER_NAME:-' + app + '}" '
        '&& echo "$DEPLOY_KEY" > ~/.ssh/"$name" && chmod 600 ~/.ssh/"$name"')
    job = wfb.job(job_id, needs=opts.get('needs') or None).runs_on('ubuntu-latest').env(**env)
    step = job.checkout()
    if opts.get('lfs', True): step = step.with_(lfs=True)
    (step.end_step().setup_uv().with_(python_version='3.13').end_step()
        .uv_install('uv sync --group dev').end_step()
        .step('Install SSH key').if_("env.DEPLOY_KEY != ''").run(ssh).end_step()
        .step('Deploy').run(str(opts.get('cmd') or 'python deploy.py deploy')).end_job())

Every key of `schema` becomes an `env:` entry. A `None` value renders `secrets.KEY` and any other
value renders `vars.KEY`, which is gheasy's own `env_schema` convention.

`DEPLOY_KEY` is added whether or not the schema names it. The step that writes it is guarded by
`if: env.DEPLOY_KEY != ''`, so a checkout without the secret runs the rest of the job instead of
failing in the middle of it. The key is written to `~/.ssh/$SERVER_NAME`, falling back to the
application name. LFS files are fetched unless the spec turns them off.

In [ ]:
wfb = Workflow('deploy')
deploy_job(wfb, 'deploy', {'cmd': 'python deploy.py deploy'}, {'API_KEY': None, 'REGION': 'eu'}, 'lego')
built(wfb)['jobs']['deploy']['env']

{'API_KEY': '${{ secrets.API_KEY }}',
 'REGION': '${{ vars.REGION }}',
 'DEPLOY_KEY': '${{ secrets.DEPLOY_KEY }}'}

In [ ]:
#| hide
job = built(wfb)['jobs']['deploy']
test_eq([s['name'] for s in job['steps']],
    ['Checkout', 'Setup uv', 'Install dependencies', 'Install SSH key', 'Deploy'])
test_eq(job['steps'][0]['with'], {'lfs': True})
test_eq(job['steps'][3]['if'], "env.DEPLOY_KEY != ''")
assert 'SERVER_NAME:-lego' in job['steps'][3]['run']
bare = Workflow('deploy'); deploy_job(bare, 'deploy', {'lfs': False}, None, 'app')
job = built(bare)['jobs']['deploy']
test_eq(job['env'], {'DEPLOY_KEY': '${{ secrets.DEPLOY_KEY }}'})
assert 'with' not in job['steps'][0], 'lfs off asks checkout for nothing'
test_eq(job['steps'][-1]['run'], 'python deploy.py deploy')

In [ ]:
#| export
def fastship_job(wfb, job_id='release', needs=None):
    "The release job a fastship tag push expects, from gheasy's own preset."
    if not hasattr(wfb, 'fastship_release_job'):
        raise AttributeError('this gheasy has no fastship release job: pip install -U gheasy')
    wfb.fastship_release_job(needs=needs, job_id=job_id)

The release job belongs to gheasy, not here. A gheasy without `fastship_release_job` raises rather
than building a workflow that is missing its release job, and the message names the upgrade.

In [ ]:
wfb = Workflow('release')
if hasattr(wfb, 'fastship_release_job'):
    fastship_job(wfb)
    test_eq(list(built(wfb)['jobs']), ['release'])
else: test_fail(lambda: fastship_job(wfb), contains='pip install -U gheasy')

In [ ]:
#| export
def nbdev_job(wfb, job_id='test', needs=None):
    "nbdev's own CI job: export and test, then the clean check that is what breaks in an nbdev repo."
    (wfb.job(job_id, needs=needs).runs_on('ubuntu-latest')
        .checkout().end_step().setup_uv().end_step().uv_install().end_step()
        .uv_run('nbdev_test').end_step()
        .step('Notebooks are clean').run('uv run nbdev_clean --check_all').end_job())

nbdev's own CI job. `nbdev_test` runs the notebooks, and `nbdev_clean --check_all` fails when a
notebook still carries the output and metadata that make its diffs unreadable.

In [ ]:
wfb = Workflow('ci'); nbdev_job(wfb)
[s.get('run') or s['uses'] for s in built(wfb)['jobs']['test']['steps']]

['actions/checkout@v4',
 'astral-sh/setup-uv@v5',
 'uv sync --frozen',
 'uv run nbdev_test',
 'uv run nbdev_clean --check_all']

In [ ]:
#| export
def _run_job(wfb, job_id, opts):
    job = wfb.job(job_id, needs=opts.get('needs') or None).runs_on('ubuntu-latest')
    step = job.checkout().end_step().setup_uv().end_step()
    if opts.get('install', True): step = step.uv_install().end_step()
    cmds = _lines(opts.get('cmds')) or ['echo hello']
    for i, cmd in enumerate(cmds):
        step = step.step(f'Step {i + 1}').run(cmd)
        step = step.end_step() if i + 1 < len(cmds) else step
    step.end_job()

The `run` preset is the general case: one step per line of `cmds`, named `Step N` by position.
Blank lines are dropped. Empty `cmds` builds `echo hello` rather than a job with no steps, which
GitHub rejects. `install` off leaves checkout and setup-uv in place and drops the `uv sync` step.

In [ ]:
wfb = Workflow('ci')
_run_job(wfb, 'smoke', {'cmds': 'echo one\n\necho two', 'install': False})
[(s['name'], s.get('run') or s['uses']) for s in built(wfb)['jobs']['smoke']['steps']]

[('Checkout', 'actions/checkout@v4'),
 ('Setup uv', 'astral-sh/setup-uv@v5'),
 ('Step 1', 'echo one'),
 ('Step 2', 'echo two')]

In [ ]:
#| hide
wfb = Workflow('ci'); _run_job(wfb, 'run', {'needs': ['test']})
job = built(wfb)['jobs']['run']
test_eq(job['needs'], ['test'])
test_eq([s['name'] for s in job['steps']], ['Checkout', 'Setup uv', 'Install dependencies', 'Step 1'])
test_eq(job['steps'][-1]['run'], 'echo hello')

gheasy's job presets each name their own job: `uv_test_job` always builds `test`, `rust_job` and
`go_job` both build `build`. Three of them take no `needs` at all. `_adopt` puts the id `compose`
worked out onto the job that was just added, and the `needs` the spec asked for with it. Two rows
of the same preset are then two jobs, not one job silently overwriting the other.

In [ ]:
#| export
def _adopt(wfb, job_id, needs):
    "Put compose's id and needs onto the job a gheasy preset just added under its own."
    job = wfb._jobs[-1]
    wfb._job_map.pop(job._id, None)
    job._id, wfb._job_map[job_id] = job_id, job
    if needs: job._data['needs'] = needs

In [ ]:
#| export
def compose(spec, schema=None, app=''):
    "A spec as YAML, built entirely with gheasy. Job ids are made unique as they are added."
    from gheasy.workflow import Workflow
    spec = spec or {}
    app = str(app or spec.get('app') or 'app')
    wfb = Workflow(str(spec.get('name') or 'ci'))
    _triggers(wfb, spec.get('on') or {})
    by_id = {p['id']: p for p in PRESETS}
    used = set()
    for row in spec.get('jobs') or ():
        preset = by_id.get(str((row or {}).get('preset') or ''))
        if preset is None: continue
        opts = dict(row.get('options') or {})
        needs = [n for n in _listy(row.get('needs')) if n]
        opts['needs'] = needs or None
        job_id = str(row.get('id') or opts.get('job_id') or preset['job'])
        n = 2
        while job_id in used: job_id, n = f"{preset['job']}{n}", n + 1
        used.add(job_id)
        pid = preset['id']
        if pid == 'uv_test':
            wfb.uv_test_job(test_cmd=str(opts.get('test_cmd') or 'pytest'), needs=opts['needs'],
                python_versions=_listy(opts.get('python_versions')) or None)
        elif pid == 'uv_lint':
            wfb.uv_lint_job(lint_cmd=str(opts.get('lint_cmd') or
                'uv run ruff check . && uv run ruff format --check .'),
                needs=opts['needs'])
        elif pid == 'uv_pypi': wfb.uv_pypi_job(needs=opts['needs'])
        elif pid == 'docker': wfb.docker_job(str(opts.get('app_name') or app), needs=opts['needs'])
        elif pid == 'node': wfb.node_job()
        elif pid == 'rust': wfb.rust_job()
        elif pid == 'go': wfb.go_job()
        elif pid == 'nbdev_test': nbdev_job(wfb, job_id, opts['needs'])
        elif pid == 'fastship': fastship_job(wfb, job_id, opts['needs'])
        elif pid == 'deploy': deploy_job(wfb, job_id, opts, schema, app)
        elif pid == 'run': _run_job(wfb, job_id, opts)
        _adopt(wfb, job_id, opts['needs'])
    return wfb.build().to_yaml()

`compose` returns YAML text. It writes nothing and reads nothing from disk.

A row naming a preset that is not in `PRESETS` is skipped, so a spec written against a newer
version of this module still builds the jobs this one knows. Job ids are unique: a second row of
the same preset takes the preset's job name with a number after it, and `needs` names those ids.

`schema` and `app` come from the project, not from the spec. Only the `deploy` and `docker` presets
read them.

In [ ]:
spec = {'name': 'ci', 'on': {'push': 'main', 'pull_request': 'main'},
    'jobs': [{'preset': 'uv_lint'},
        {'preset': 'uv_test', 'needs': 'lint',
            'options': {'test_cmd': 'pytest -q', 'python_versions': '3.11, 3.12'}}]}
print(compose(spec))

name: ci
on:
  push:
    branches:
      - main
  pull_request:
    branches:
      - main
jobs:
  lint:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout
        uses: actions/checkout@v4
      - name: Setup uv
        uses: astral-sh/setup-uv@v5
      - name: Install dependencies
        run: uv sync --frozen
      - name: Lint
        run: uv run ruff check . && uv run ruff format --check .
  test:
    needs:
      - lint
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version:
          - '3.11'
          - '3.12'
    steps:
      - name: Checkout
        uses: actions/checkout@v4
      - name: Setup uv
        uses: astral-sh/setup-uv@v5
      - name: Install dependencies
        run: uv sync --frozen
      - name: Test
        run: uv run pytest -q



In [ ]:
#| hide
doc = parse_wf(compose(spec))
test_eq(list(doc['jobs']), ['lint', 'test'])
test_eq(doc['jobs']['test']['needs'], ['lint'])
test_eq(doc['jobs']['test']['strategy']['matrix']['python-version'], ['3.11', '3.12'])
test_eq(doc['on'], {'push': {'branches': ['main']}, 'pull_request': {'branches': ['main']}})
test_eq(parse_wf(compose({'jobs': [{'preset': 'unknown'}, {}]}))['jobs'], {})
test_eq(parse_wf(compose(None))['name'], 'ci')

In [ ]:
#| hide
every = [p['id'] for p in PRESETS if p['id'] != 'fastship']   # fastship needs a newer gheasy
jobs = parse_wf(compose({'jobs': [{'preset': p} for p in every]}))['jobs']
test_eq(len(jobs), len(every))                                # every preset builds exactly one job
assert all(j.get('steps') for j in jobs.values()), 'and no job is built without steps'
two = parse_wf(compose({'jobs': [{'preset': 'uv_test'},
    {'preset': 'node', 'needs': 'test'}, {'preset': 'uv_test', 'needs': 'frontend'}]}))['jobs']
test_eq(list(two), ['test', 'frontend', 'test2'])
test_eq(two['frontend']['needs'], ['test'])                   # node_job itself takes no needs
test_eq(two['test2']['needs'], ['frontend'])
test_eq(parse_wf(compose({'jobs': [{'preset': 'docker'}]}, app='lego'))['jobs']['docker']
    ['steps'][-1]['with']['tags'], 'ghcr.io/${{ github.repository_owner }}/lego:latest')

In [ ]:
#| export
def spec_path(root, file, dir=None):
    return Path(root)/(dir or _spec_dir)/Path(file).with_suffix('.json')

In [ ]:
#| export
def load_spec(root, file, dir=None):
    "The spec a workflow was built from, or None when it was written some other way."
    try: return json.loads(spec_path(root, file, dir).read_text(encoding='utf-8'))
    except (OSError, ValueError): return None

In [ ]:
#| export
def save_spec(root, file, spec, dir=None):
    p = spec_path(root, file, dir)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(spec, indent=2) + '\n', encoding='utf-8')
    return p

`save_spec` writes the spec a workflow was built from beside the workflow's own name: `ci.yml`
becomes `workflows/ci.json`. It creates the folder if it is missing and returns the path it wrote.

`load_spec` returns `None` for a workflow with no spec and for a spec that will not parse. A
workflow written by hand reads as one without a spec, never as an error.

In [ ]:
tmp = TemporaryDirectory(); root = Path(tmp.name)
p = save_spec(root, 'ci.yml', spec)
p.relative_to(root), load_spec(root, 'ci.yml') == spec

(Path('workflows/ci.json'), True)

In [ ]:
#| hide
test_eq(spec_path('/r', 'ci.yaml'), Path('/r/workflows/ci.json'))
test_eq(spec_path('/r', 'ci'), Path('/r/workflows/ci.json'))
test_eq(spec_path('/r', 'ci', '.leela/workflows'), Path('/r/.leela/workflows/ci.json'))
use_spec_dir('.leela/workflows')
try:
    test_eq(spec_dir(), '.leela/workflows')
    test_eq(spec_path('/r', 'ci'), Path('/r/.leela/workflows/ci.json'))
    test_eq(save_spec(root, 'hosted.yml', spec), root/'.leela'/'workflows'/'hosted.json')
    test_eq(load_spec(root, 'hosted.yml'), spec)
finally: use_spec_dir(SPEC_DIR)
test_eq(load_spec(root, 'hosted.yml'), None)                  # the default cannot see the host's
test_eq(load_spec(root, 'hosted.yml', '.leela/workflows'), spec)
test_eq(load_spec(root, 'missing.yml'), None)
(root/'workflows'/'bad.json').write_text('{ not json')
test_eq(load_spec(root, 'bad.yml'), None)
tmp.cleanup()